# 07 · Launch the application

**Purpose** — start the Gradio front end for the live demo.

**Prerequisites** — notebook 01 has been run so the database exists. The app runs
on the deterministic engine alone, so it works **with or without** the NIMs up;
the voice and agent surfaces activate once they are. For the semantic tab and
grounded narration you also want 00 (NIMs) and 04 (the index).

> This cell **blocks the kernel for as long as the app is up** — that is how it
> stays serving. Run it last, in its own notebook, and use Kernel -> Interrupt
> to stop the server.

### Tabs
| Tab | What it shows |
|---|---|
| Shop Floor | Every RO, filterable by blocked / at-risk / safety / waiter |
| Repair Order | Derived state plus the update history as written |
| Technician Update | Log work, then see the **diff card** of what changed |
| Shift Handover | Prioritised brief: safety, breached, at-risk, blocked |
| Insights | Cross-RO patterns a person scanning one RO at a time would miss |

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.abspath(".."))          # notebooks/ -> repo root
os.chdir(os.path.abspath(".."))
from dotenv import load_dotenv; load_dotenv()
print("repo root:", os.getcwd())

## Config

Forward port 7860 over SSH and open http://127.0.0.1:7860 — that is the default
below. `share=True` publishes a world-reachable `gradio.live` link instead, and
the Technician Update tab **writes to the event log**, so set `GRADIO_AUTH` in
`.env` before you turn it on.

**"now" is not pinned here, deliberately.** The clock follows the newest event
in the log, so a dataset is never stale. This cell used to hard-code
`ASOIA_NOW = "2026-09-24T16:55:00"`; against a dataset generated today that is a
moment weeks before the data, so every time-window question — the handover,
"this week", "overnight" — returns nothing, cites nothing, and is refused by the
output rail. The demo looked broken and the data was fine.

In [ ]:
PORT = 7860
SHARE = False         # True publishes a public link; set GRADIO_AUTH in .env first
# os.environ["ASOIA_NOW"] = "2026-09-24T16:55:00"   # pin the clock, to reproduce one run
from app.state import clock
print("clock:", clock.source())

## Execute

In [ ]:
from app.ui.gradio_app import build
demo = build()
demo.launch(server_name="0.0.0.0", server_port=PORT, share=SHARE)

## What you should see

A public Gradio URL. Demo path that lands well:
1. **Shop Floor** → filter `safety` — the vehicles that cannot be released
2. **Shift Handover** → Generate — safety first, each with a concrete next action
3. **Insights** → one part blocking several ROs; stalled and authorisation-delayed work
4. **Technician Update** → log an operation and watch the **diff card** react